# Evidence RAG v1 - Testing and Fine-Tuning

This notebook demonstrates strategies for testing the chatbot and for fine-tuning both the embedding model and the generator (inference) model.

In [ ]:
# Install and import libraries

%pip install requests sentence-transformers transformers jsonref
%pip install ragas datasets langchain-google-genai langchain-community
%pip install peft accelerate

import requests
import re
import io
import os
import time
import pickle
import torch
import textwrap
import gc
import numpy as np
from google import genai
from google.colab import drive, userdata
from pathlib import Path
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
from transformers import T5ForConditionalGeneration, AutoTokenizer
from ragas import evaluate
from ragas.llms import llm_factory
from ragas.metrics import AnswerCorrectness, ContextPrecision, ContextRecall, Faithfulness
from ragas.embeddings import GoogleEmbeddings
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

In [ ]:
# Gemini API setup

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"),
    http_options=genai.types.HttpOptions(
        timeout=60000,  # milliseconds, so this is 60 seconds
        retry_options={
            "attempts": 3,
            "initial_delay": 1.0
        }
    )
)

llm = llm_factory(
    "gemini-3.1-flash-lite-preview",
    provider="google",
    client=client
)

In [5]:
# Load data from storage

print("Loading data from storage")
try:
    drive.mount('/content/drive')
    with open('/content/drive/MyDrive/rag_corpus.pkl', 'rb') as f:
        corpus = pickle.load(f)
    with open('/content/drive/MyDrive/rag_documents.pkl', 'rb') as f:
        documents = pickle.load(f)
    with open('/content/drive/MyDrive/rag_papers.pkl', 'rb') as f:
        papers = pickle.load(f)
    with open('/content/drive/MyDrive/rag_vectors.pkl', 'rb') as f:
        vectors = pickle.load(f)
    print("\nLoaded", len(documents), "guidelines,", len(papers), "review articles,", len(corpus), "chunks and", len(vectors), "vectors")
except Exception as e:
    raise(e)

Loading data from storage
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Loaded 880 guidelines, 1003 review articles, 60201 chunks and 60201 vectors


In [6]:
# Embedding setup

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

def embed_prog(texts):
    return np.array(embed_model.encode(texts, show_progress_bar=True))

def embed(texts):
    return np.array(embed_model.encode(texts, show_progress_bar=False))

def new_embed(texts):
    return np.array(new_embed_model.encode(texts, show_progress_bar=False))

def format_chunk(chunk):
    title = chunk["title"]
    heading = chunk["heading"]
    return f"""
Title: {title}
Section: {heading}
Content:
{chunk['text']}
""".strip()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
# Retrieval

def search(query, k=10, threshold=0.4, version="old"):
    if version == "new":
        q = new_embed([query])[0]
    else:
        q = embed([query])[0]
    sims = vectors @ q / (
        np.linalg.norm(vectors, axis=1) * np.linalg.norm(q)
    )
    idx = np.argsort(-sims)
    results = []
    for i in idx:
        if sims[i] < threshold:
            continue
        results.append((corpus[i], sims[i], i))
        if len(results) >= k:
            break
    return results

In [8]:
# Query function

prompt_start = "Use the medical evidence below to answer. Construct an answer as a short sentence or list of points. Don't just quote."

def ask(question):
    results = search(question)
    if len(results) == 0:
        print("\nNo relevant sources found")
        return None
    print("Returned", len(results), "chunks:")
    for r in results:
        d, sim, i = r
        print("\nSource:", d["source"], end="")
        if d["title"]:
            print(":", d["title"])
        else:
            print("")
        print("Similarity score: [" + str(round(sim, 3)) + "]")
        print("Chunk:")
        print(textwrap.fill(format_chunk(d), width=80))
    context_str = "\n\n".join(format_chunk(r[0]) for r in results)
    prompt = f"""

{prompt_start}

Context:
{context_str}

Question:
{question}
"""

    input_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).input_ids.to(model.device)
    outputs = model.generate(input_ids, max_new_tokens=256)
    raw = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("\nAnswer:\n")
    print(textwrap.fill(raw, width=80))
    time.sleep(1)
    ground_truth = input("\nEnter a ground truth response for this question: ")
    questions.append(question)
    context = [format_chunk(r[0]) for r in results]
    contexts.append(context)
    answers.append(raw)
    ground_truths.append(ground_truth)

The flan-t5-large model from Hugging Face is used for inference.

In [9]:
# Model setup

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-large")
model = model.to("cuda" if torch.cuda.is_available() else "cpu")

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
# Test parameters

questions = []
answers = []
contexts = []
ground_truths = []

In [ ]:
# List current guidelines

for document in documents:
    print(document["url"], end=" - ")
    print(document["title"])

In [ ]:
# List current review articles

for paper in papers:
    print(paper["url"], end=" - ")
    print(paper["title"])

### Compiling test data

To compile the test data, the user enters questions and a ground truth answer for each question.

In [ ]:
# Test data compilation

print("Answers given by this chatbot are AI-generated and MAY CONTAIN ERRORS.")
print("Please check the provided sources.")

while True:
    if len(questions) == 1:
        print("\nTest data compiled for 1 question.")
    else:
        print("\nTest data compiled for", len(questions), "questions.")
    question = input("\n\nAsk a question: ")
    if question.lower() == "break":
        break
    ask(question)

Answers given by this chatbot are AI-generated and MAY CONTAIN ERRORS.
Please check the provided sources.

Test data compiled for 0 questions.


Ask a question: How do you treat febrile convulsions?
Returned 10 chunks:

Source: https://www.rch.org.au/clinicalguide/guideline_index/Febrile_seizure/: Clinical Practice Guidelines : Febrile seizure
Similarity score: [0.556]
Chunk:
Title: Clinical Practice Guidelines : Febrile seizure Section: Contact us
Content: 50 Flemington Road ParkvilleVictoria 3052 Australia +61 3 9345 5522

Source: https://www.rch.org.au/clinicalguide/guideline_index/Febrile_seizure/: Clinical Practice Guidelines : Febrile seizure
Similarity score: [0.544]
Chunk:
Title: Clinical Practice Guidelines : Febrile seizure Section: 5 minutes
Content: Manage the underlying cause of the feverChildren with simple febrile
seizures have the equivalent risk of serious bacterial infection as those with
fever alone Ongoing treatment with antiepileptic drugs is not routinely
recommen

In [ ]:
# Create client with API key
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"),
    http_options=genai.types.HttpOptions(
        timeout=60000,  # milliseconds, so this is 60 seconds
        retry_options={
            "attempts": 3,
            "initial_delay": 1.0
        }
    )
)

# Create LLM - adapter is auto-detected for google provider
llm = llm_factory(
    "gemini-3.1-flash-lite-preview",
    provider="google",
    client=client
)

### Testing with RAGAS

[RAGAS](https://docs.ragas.io/en/stable/) (Retrieval Augmented Generation Assessment) is used for testing, based on the following metrics:
- Context Precision - Relevance of the retrieved chunks to the question
- Context Recall - Relevance of the retrieved chunks to the ground truth answer
- Faithfulness - How much of the generated answer was in the retrieved chunks (without additions or hallucinations)
- Answer Correctness - How close the generated answer is to the ground truth answer

In [ ]:
# Execute RAGAS test

test_data = {
    "question": questions,
    "answer": answers,
    "contexts": contexts,
    "ground_truth": ground_truths
}

dataset = Dataset.from_dict(test_data)
embeddings = GoogleEmbeddings(client=client, model="gemini-embedding-001")

test_results = evaluate(
    dataset,
    metrics = [
        ContextPrecision(llm=llm),
        ContextRecall(llm=llm),
        Faithfulness(llm=llm),
        AnswerCorrectness(llm=llm, embeddings=embeddings)
    ],
    allow_nest_asyncio=False
)

print("RESULTS:", test_results)

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

RESULTS: {'context_precision': 0.5500, 'context_recall': 1.0000, 'faithfulness': 0.7857, 'answer_correctness': 0.5183}


Context precision of 0.55 suggests that nearly half of the retrieved chunks are irrelevant.

Context recall of 100% is expected given that this was not a blind test and ground truth answers were based on the displayed chunks.

Faithfulness of 0.79 suggests that around 20% of the generated answer content is based on the model's own training rather than the retrieved chunks. However, close inspection of the test data compilation output shows that this is not the case. *All* of the generated answers are fully derived from the chunks, although in some cases chunk sections are inappropriately joined together to create incorrect answers - e.g. "The most common cause of otalgia is aphthous ulceration and malignancy", which is not true. This suggests that the faithfulness metric of this package should be interpreted with some caution.

Answer correctness of 0.52 is consistent with inspection of the compiled testing data.
- Febrile convulsion and pyloric stenosis questions - The generated answers are acceptable but incomplete.
- Myocardial infarction question - *Most* of the generated answer is factually correct, but most of it is irrelevant to the question about "features", which for a clinician would imply symptoms, signs and investigation findings.
- Earache question - The answer is clearly wrong. The model has joined up two chunk fragments - "most common cause" and "aphthous ulceration and malignancy" - but failed to recognise that this creates a new meaning that is not found anywhere in the retrieved chunks.
- Melanoma question - The answer is correct.

A limitation of this testing process is that each question could have a wide variety of acceptable answers, so blind testing would allow for a more meaningful measurement of context recall (relevance of chunks to the ground truth answer) but would introduce much more subjectivity into the answer correctness metric. Non-blind testing as shown here allows for a subjective assessment of answer correctness with the addition of a quantifiable metric.

A good starting point for improvement would be to increase the quality of retrieved chunks through fine-tuning the embedding model. Improving answer correctness might require fine-tuning of the generator (inference) model with a very large dataset, or substitution with a higher quality model.

### Fine-tuning the embedding model

This involves inspecting the retrieved chunks for a set of questions, and for each question nominating a "good" chunk and a "bad" chunk, either from the chunks provided or by entering free text. "Skip" can be entered if all the chunks look good.

In [ ]:
# Embed fine-tuning setup

def add_example(question):
    results = search(question)
    if len(results) == 0:
        print("\nNo relevant sources found")
        return None
    print("Returned", len(results), "chunks:")
    j = 0
    for r in results:
        d, sim, _ = r
        print("\nCHUNK " + str(j) + ":")
        print("Source:", d["source"], end="")
        if d["title"]:
            print(":", d["title"])
        else:
            print("")
        print("Similarity score: [" + str(round(sim, 3)) + "]")
        print("Chunk:")
        print(textwrap.fill(format_chunk(d), width=80))
        j += 1
    time.sleep(1)
    good_chunk = input("\nEnter a good chunk (0-9 from above or free text or 'skip'): ")
    if good_chunk.lower() == "skip":
        return None
    if good_chunk.isdigit():
        n = int(good_chunk)
        if n >= 0 and n < len(results):
            good_chunk = results[n][0]["text"]
    bad_chunk = input("\nEnter a bad chunk (0-9 from above or free text): ")
    if bad_chunk.isdigit():
        n = int(bad_chunk)
        if n >= 0 and n < len(results):
            bad_chunk = results[n][0]["text"]
    return [question, good_chunk, bad_chunk]

In [ ]:
examples = []

In [ ]:
new_embed_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Compile embedding examples

while True:
    if len(examples) == 1:
        print("\nEmbedding examples compiled for 1 question.")
    else:
        print("\nEmbedding examples compiled for", len(examples), "questions.")
    question = input("\n\nAsk a question: ")
    if question.lower() == "break":
        break
    example = add_example(question)
    if example:
        examples.append(InputExample(texts=example))


Embedding examples compiled for 8 questions.


Ask a question: What are some retinal pathologies?
Returned 10 chunks:

CHUNK 0:
Source: https://pmc.ncbi.nlm.nih.gov/articles/PMC8882446: Voretigene neparvovec for inherited retinal dystrophy - PMC
Similarity score: [0.458]
Chunk:
Title: Voretigene neparvovec for inherited retinal dystrophy - PMC Section:
ACTIONS Content: View on publisher site PDF (61.5 KB) Cite Collections
PermalinkPERMALINKCopy

CHUNK 1:
Source: https://pmc.ncbi.nlm.nih.gov/articles/PMC8762590: Visual Phenomena Associated With Migraine and Their Differential Diagnosis - PMC
Similarity score: [0.447]
Chunk:
Title: Visual Phenomena Associated With Migraine and Their Differential
Diagnosis - PMC Section: If a patient’s visual symptoms remain present when each
eye is covered in turn, where is the pathology presumably to be found? Content:
symptoms is most likely to indicate the presence of optic neuritis? Question 10
A patient describes seeing “tracks”: “A fast-moving bal

The output above is kept brief for display purposes. As shown, 8 examples were provided previously. The questions asked for these two examples were deliberately vague.

In [ ]:
# Fine-tune embedding model

loader = DataLoader(examples, shuffle=True, batch_size=16)
loss = losses.MultipleNegativesRankingLoss(new_embed_model)
new_embed_model.fit(train_objectives=[(loader, loss)], epochs=3)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


In [ ]:
# Compare embeddings function

def compare_embeds(question, show_chunks=True, show_answer=True):
    old_results = search(question)
    print("\nWith old embeddings:")
    print("\nReturned", len(old_results), "chunks:")
    print([(int(i), round(float(s), 7)) for (_, s, i) in old_results])
    if show_chunks:
        for r in old_results:
            d, sim, i = r
            print("\nSource:", d["source"], end="")
            if d["title"]:
                print(":", d["title"])
            else:
                print("")
            print("Similarity score: [" + str(round(sim, 3)) + "]")
            print("Chunk:")
            print(textwrap.fill(format_chunk(d), width=80))
    if show_answer:
        context_str = "\n\n".join(format_chunk(r[0]) for r in old_results)
        prompt = f"""

{prompt_start}

Context:
{context_str}

Question:
{question}
"""

        input_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).input_ids.to(model.device)
        outputs = model.generate(input_ids, max_new_tokens=256)
        raw = tokenizer.decode(outputs[0], skip_special_tokens=True)
        print("\nAnswer:\n")
        print(textwrap.fill(raw, width=80))

    new_results = search(question, version="new")
    print("\nWith new embeddings:")
    print("\nReturned", len(new_results), "chunks:")
    print([(int(i), round(float(s), 7)) for (_, s, i) in new_results])
    if show_chunks:
        for r in new_results:
            d, sim, i = r
            print("\nSource:", d["source"], end="")
            if d["title"]:
                print(":", d["title"])
            else:
                print("")
            print("Similarity score: [" + str(round(sim, 3)) + "]")
            print("Chunk:")
            print(textwrap.fill(format_chunk(d), width=80))
    if show_answer:
        context_str = "\n\n".join(format_chunk(r[0]) for r in new_results)
        prompt = f"""

{prompt_start}

Context:
{context_str}

Question:
{question}
"""

        input_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).input_ids.to(model.device)
        outputs = model.generate(input_ids, max_new_tokens=256)
        raw = tokenizer.decode(outputs[0], skip_special_tokens=True)
        print("\nAnswer:\n")
        print(textwrap.fill(raw, width=80))

In [ ]:
# Compare embedding models

while True:
    question = input("\n\nAsk a question: ")
    if question.lower() == "break":
        break
    compare_embeds(question, show_chunks=False, show_answer=False)



Ask a question: How do you treat febrile convulsions?

With old embeddings:

Returned 10 chunks:
[(41395, 0.5556004), (41388, 0.5435818), (41387, 0.5423837), (41389, 0.5387505), (41385, 0.530935), (41379, 0.5290768), (41380, 0.5290768), (39831, 0.5253726), (41386, 0.5242842), (39832, 0.506172)]

With new embeddings:

Returned 10 chunks:
[(41395, 0.5555989), (41388, 0.5435807), (41387, 0.5423829), (41389, 0.5387486), (41385, 0.5309331), (41379, 0.5290754), (41380, 0.5290754), (39831, 0.5253722), (41386, 0.5242822), (39832, 0.5061715)]


Ask a question: Does climate change affect blood pressure?

With old embeddings:

Returned 10 chunks:
[(26131, 0.5466678), (26108, 0.5404729), (26106, 0.5399466), (26237, 0.5392091), (26162, 0.5392091), (26233, 0.5381014), (26158, 0.5381014), (26112, 0.5316654), (26141, 0.528077), (26105, 0.5273185)]

With new embeddings:

Returned 10 chunks:
[(26131, 0.5466694), (26108, 0.5404743), (26106, 0.539948), (26162, 0.53921), (26237, 0.53921), (26233, 0.53810

In the display above, a set of chunk IDs is shown for each question using the original embedding model and the fine-tuned embedding model, along with a similarity score for each chunk. As shown, the same set of chunks is returned in each case but the order of chunks is shifted slightly for the climate change question. This suggests that a much larger dataset would be needed for fine-tuning in order to produce any meaningful change.

### Fine-tuning the generator

This process uses Low-Rank Adaption ([LoRA](https://huggingface.co/docs/peft/package_reference/lora)) to freeze the original model weights and add a much smaller set of weights for fine-tuning, as an alternative to having to update all 800 million parameters in the flan-t5-large model in this case, or billions of parameters for the most commonly used models. Data compilation involves entering questions and a ground truth answer for each question.

In [10]:
def add_gen_example(question, show_chunks=False):
    results = search(question)
    if len(results) == 0:
        print("\nNo relevant sources found")
        return None
    if show_chunks:
        print("\nReturned", len(results), "chunks:")
        for r in results:
            d, sim, i = r
            print("\nSource:", d["source"], end="")
            if d["title"]:
                print(":", d["title"])
            else:
                print("")
            print("Similarity score: [" + str(round(sim, 3)) + "]")
            print("Chunk:")
            print(textwrap.fill(format_chunk(d), width=80))
    context_str = "\n\n".join(format_chunk(r[0]) for r in results)
    prompt = f"""

{prompt_start}

Context:
{context_str}

Question:
{question}
"""

    ground_truth = input("\nEnter a ground truth response for this question: ")
    gen_questions.append(question)
    training_examples.append({"input": prompt, "target": ground_truth})

In [11]:
training_examples = []
gen_questions = []

In [12]:
new_model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-large")
new_model = new_model.to("cuda" if torch.cuda.is_available() else "cpu")

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [13]:
# Compile generator training examples

while True:
    if len(training_examples) == 1:
        print("\nGenerator examples compiled for 1 question.")
    else:
        print("\nGenerator examples compiled for", len(training_examples), "questions.")
    question = input("\n\nAsk a question: ")
    if question.lower() == "break":
        break
    example = add_gen_example(question)
    if example:
        training_examples.append(example)


Generator examples compiled for 0 questions.


Ask a question: How do you treat febrile convulsions?

Enter a ground truth response for this question: Address the cause of the fever. Treat the seizure if it is still ongoing after 5 minutes.

Generator examples compiled for 1 question.


Ask a question: How do you treat pyloric stenosis?

Enter a ground truth response for this question: Perform a pyloromyotomy once the patient’s clinical status has normalized, specifically ensuring serum bicarbonate levels have stabilized to prevent post-operative breathing complications.

Generator examples compiled for 2 questions.


Ask a question: What are the features of a myocardial infarction?

Enter a ground truth response for this question: A diagnosis of myocardial infarction requires elevated troponins in conjunction with a clinical history consistent with myocardial ischaemia, ischaemic changes on the ECG, or ancillary evidence of coronary ischaemia on cardiac imaging if available.

Generat

In [ ]:
# Prepare dataset and LoRA adaptor

dataset = Dataset.from_list(training_examples)
train_test = dataset.train_test_split(test_size=0.2)

lora_config = LoraConfig(
    r=8,                     # optimised for CPU
    lora_alpha=16,
    target_modules=["q", "v"],  # T5 attention module names
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

new_model = get_peft_model(new_model, lora_config)
new_model.print_trainable_parameters()
# Expect something like: trainable params: 589,824 || all params: 783,150,080 || trainable%: 0.075

In [ ]:
# Tokenise the dataset

def tokenize(examples):
    inputs = tokenizer(
        examples["input"],
        max_length=512,
        truncation=True,
        padding=False
    )
    targets = tokenizer(
        examples["target"],
        max_length=128,
        truncation=True,
        padding=False
    )
    # Replace padding token ids in labels with -100
    # so they're ignored in the loss calculation
    labels = [
        [(t if t != tokenizer.pad_token_id else -100) for t in target]
        for target in targets["input_ids"]
    ]
    inputs["labels"] = labels
    return inputs

tokenized = train_test.map(tokenize, batched=True, remove_columns=["input", "target"])

In [ ]:
# Train

gc.collect()
torch.cuda.empty_cache()
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

training_args = TrainingArguments(
    #output_dir="/content/drive/MyDrive/flan_t5_lora",
    num_train_epochs=5,          # More epochs to compensate for small dataset
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,  # Effective batch size = 4
    warmup_steps=10,
    learning_rate=3e-4,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    #predict_with_generate=True,  # Required for seq2seq evaluation
    fp16=False,                  # No fp16 on CPU
    report_to="none"
)

trainer = Trainer(
    model=new_model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    #tokenizer=tokenizer,
    data_collator=data_collator
)

trainer.train()

In [18]:
# Compare answers using training questions

for i in range(len(gen_questions)):
    print("\n\nQuestion:",gen_questions[i])
    prompt = training_examples[i]["input"]
    input_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).input_ids.to(model.device)

    # Using original model
    outputs = model.generate(input_ids, max_new_tokens=256)
    raw = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("\nAnswer using original model:\n")
    print(textwrap.fill(raw, width=80))

    # Using fine-tuned model
    outputs = new_model.generate(input_ids=input_ids, max_new_tokens=256)
    raw = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("\nAnswer using fine-tuned model:\n")
    print(textwrap.fill(raw, width=80))



Question: How do you treat febrile convulsions?

Answer using original model:

Treat the seizure if duration ongoing after5 minutes.

Answer using fine-tuned model:

Treat the seizure if duration ongoing after5 minutes.


Question: How do you treat pyloric stenosis?

Answer using original model:

Definitive treatment is by pyloromyotomy.

Answer using fine-tuned model:

Definitive treatment is by pyloromyotomy.


Question: What are the features of a myocardial infarction?

Answer using original model:

Non-STEMI is more complex to establish, due to the rising incidence of non-type
1 myocardial infarctions and myocardial injuries. Interpretation of the complete
clinical presentation in the context of the Fourth Universal Definition of
Myocardial Infarction is recommended rather than relying on troponin elevation
alone. After a diagnosis of non-STEMI has been confirmed, acute management
includes antiplatelet therapy and anticoagulation, and coronary investigation
should be considered. 

In [19]:
# Compare answers using additional questions

def compare_generators(question):
    results = search(question)
    if len(results) == 0:
        print("\nNo relevant sources found")
        return None
    context_str = "\n\n".join(format_chunk(r[0]) for r in results)
    prompt = f"""

{prompt_start}

Context:
{context_str}

Question:
{question}
"""

    input_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).input_ids.to(model.device)

    # Using original model
    outputs = model.generate(input_ids, max_new_tokens=256)
    raw = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("\nAnswer using original model:\n")
    print(textwrap.fill(raw, width=80))

    # Using fine-tuned model
    outputs = new_model.generate(input_ids=input_ids, max_new_tokens=256)
    raw = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("\nAnswer using fine-tuned model:\n")
    print(textwrap.fill(raw, width=80))

while True:
    question = input("\n\nAsk a question: ")
    if question.lower() == "break":
        break
    compare_generators(question)



Ask a question: Which kids should get blood transfusions?

Answer using original model:

The guideline from the “European Society of Anaesthesiology and Intensive Care”
(ESAIC) for “severe perioperative bleeding” (20) (Box), revised in 2022, is
opposed to transfusion if the child is hemodynamically stable and the Hb
concentration is at least 7 g/dL.

Answer using fine-tuned model:

Approximately 1% to 2% of all hospitalized children in Germany receive a
transfusion of blood products.


Ask a question: How do you treat neuropathic pain?

Answer using original model:

Neuropathic pain may present a vast therapeutic challenge.

Answer using fine-tuned model:

Neuropathic pain may present a vast therapeutic challenge.


Ask a question: How should you prepare for a trip to Cambodia and Vietnam?

Answer using original model:

Travelers should stay hydrated, especially during midday.

Answer using fine-tuned model:

Travelers should stay hydrated, especially during midday.


Ask a question:

The two outputs above show answers produced by the original generator model and by the fine-tuned model. As seen, most of the answers are unchanged, whereas some of the changed answers might represent a slight improvement, but overall the quality of the answers is still fairly poor. This suggests that a vastly larger fine-tuning dataset would be needed for meaningful improvement.

### Conclusions

This notebook is intended to provide only a very basic demonstration of strategies for testing the chatbot and fine-tuning the models used.

RAGAS testing provides a very useful measure of appropriate chunk selection through the context recall metric. This could be used on a larger scale by generating a long set of test questions with another LLM and comparing this metric before and after fine-tuning the embedding model. As seen, the faithfulness metric is susceptible to inaccuracies but this is likely to improve once the generator model is better able to preserve meaning when interpreting chunks. Faithfulness could even be seen as a reflection of the generator's ability to make these distinctions as well as just detecting hallucinations.

Fine-tuning of the embedding model could be improved by a larger set of displayed chunks (perhaps 50) for each question as well as (of course) using a larger set of questions. However, a very large dataset may still be needed to produce a worthwhile change and there is a danger of overfitting if the dataset is not quite large enough. Another option may be to improve the quality of chunks by using sentence-aware and paragraph-aware splitting and attempting to remove chunks that contain only irrelevant information such as headings only or facility contact details.

Meaningful fine-tuning of the generator model is likely to require a very large dataset indeed, and it may be preferable just to switch to a more suitable model.